In [ ]:
#!/g/data/xp65/public/apps/med_conda_scripts/analysis3-25.07.d/bin/python3
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import xesmf as xe

from dask.distributed import Client, wait
import os, sys
import psutil

sys.path.append('/home/548/cd3022/repos/Irradiance-comparisons/Irradiance-comparisons')
import logger
from read_datasets import read_dataset

LOG = logger.get_logger(__name__)
# qsub -I -q normal -P er8 -l walltime=2:00:00,ncpus=24,mem=120GB,jobfs=100MB,storage=gdata/xp65+gdata/er8+gdata/ob53+gdata/rt52+gdata/rv74+gdata/su28

In [ ]:
client = Client(
    n_workers=48,
    threads_per_worker=1
)
client

In [ ]:
BARRA = Path('/g/data/ob53/BARRA2/output/reanalysis/')
BARRA_R2 = BARRA / "AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1"
BARRA_R2_DIR = BARRA_R2 / "1hr"

vars_to_open = [
    'clh',
    'clm',
    'cll'
]
ds_vars = []
for var in vars_to_open:

    files = BARRA_R2_DIR.glob(f'{var}/latest/*1hr_20*.nc')
    ds = xr.open_mfdataset(files, chunks='auto', concat_dim='time', combine='nested',
                          data_vars='minimal', coords='minimal', compat='override')
    ds_vars.append(ds)
barra_r2 = xr.merge(ds_vars)